# Interview AI Studio: Natural Silence Removal and Neural Speech Denoising

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RinmeSTD/DNColab/blob/main/Interview_AI_Studio.ipynb)
![GPU Acceleration](https://img.shields.io/badge/Hardware-NVIDIA%20GPU%20NVENC-green)
![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue)
![License](https://img.shields.io/badge/License-MIT-purple)

Automate video post-production for interviews, podcasts, lectures, and talking-head content:
- Silero VAD v5: Deep-learning voice activity detection with configurable padding and pause bridging.
- DeepFilterNet 3 / Resemble Enhance: State-of-the-art neural speech noise suppression.
- Loudness Normalization: EBU R128 broadcast standard (-14.0 LUFS) with true-peak limiter.
- Hardware Accelerated NVENC: Fast GPU cutting and re-encoding with CPU libx264 fallback.
- NLE Timeline Export: Final Cut Pro 7 XML (.xml) and CMX 3600 EDL (.edl) for Adobe Premiere Pro and DaVinci Resolve.


In [ ]:
# @title Step 1: Environment Setup and Hardware Check
import os
import sys
import shutil

# 1. Fetch pipeline repository if running in fresh Colab session
if not os.path.exists("interview_processor.py"):
    print("Fetching Interview Studio pipeline repository...")
    !git clone -q https://github.com/RinmeSTD/DNColab.git _repo && cp -r _repo/* . && rm -rf _repo

# 2. Download precompiled DeepFilterNet 3 standalone binary (instant ~1s, no Cargo/Rust compilation)
if not shutil.which("deep-filter"):
    print("Downloading precompiled DeepFilterNet 3 neural audio binary...")
    !curl -fsSL https://github.com/Rikorose/DeepFilterNet/releases/download/v0.5.6/deep-filter-0.5.6-x86_64-unknown-linux-musl -o /usr/local/bin/deep-filter && chmod +x /usr/local/bin/deep-filter

# 3. Install lightweight Python dependencies
print("Installing Python dependencies (soundfile, scipy, pyloudnorm, tqdm)...")
!pip install --no-warn-conflicts -q soundfile scipy pyloudnorm tqdm

# 4. Verify PyTorch and GPU runtime in active kernel
import torch
print("=" * 65)
print("System Hardware and Environment Verification:")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[GPU] Runtime Active: {device_name} ({vram_gb:.1f} GB VRAM)")
    print(f"      CUDA Driver / Runtime Version: {torch.version.cuda}")
else:
    print("[WARN] No GPU detected! Running on CPU mode.")
    print("       Tip: In Colab, go to Runtime -> Change runtime type -> T4 GPU")
print("=" * 65)
print("\nEnvironment ready for processing in seconds!")


In [ ]:
# @title Step 2: Storage and Google Drive Mount
import os

MOUNT_DRIVE = True  # @param {type:"boolean"}
DEFAULT_DRIVE_INPUT = "/content/drive/MyDrive/Interview_Studio/input"
DEFAULT_DRIVE_OUTPUT = "/content/drive/MyDrive/Interview_Studio/output"
LOCAL_INPUT = "./inputs"
LOCAL_OUTPUT = "./outputs"

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        print("Mounting Google Drive at /content/drive...")
        drive.mount("/content/drive", force_remount=False)
        resolved_input = DEFAULT_DRIVE_INPUT
        resolved_output = DEFAULT_DRIVE_OUTPUT
        print("[OK] Google Drive successfully mounted.")
    except (ImportError, ModuleNotFoundError):
        print("[INFO] Non-Colab environment detected. Using local project workspace.")
        resolved_input = LOCAL_INPUT
        resolved_output = LOCAL_OUTPUT
else:
    resolved_input = LOCAL_INPUT
    resolved_output = LOCAL_OUTPUT
os.makedirs(resolved_input, exist_ok=True)
os.makedirs(resolved_output, exist_ok=True)
os.makedirs(resolved_output, exist_ok=True)

print(f"Active Input Directory  : {os.path.abspath(resolved_input)}")
print(f"Active Output Directory : {os.path.abspath(resolved_output)}")
print("\nTip: Place your interview videos (.mp4, .mov, .mkv) in the input folder.")


In [ ]:
# @title Step 3: Processing Configuration Form { run: "auto" }

# Folder Paths
INPUT_DIR = "/content/drive/MyDrive/Interview_Studio/input"  # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/Interview_Studio/output"  # @param {type:"string"}

# Audio Denoising & Silence Cutting Parameters
DENOISE_ENGINE = "DeepFilterNet3"  # @param ["DeepFilterNet3", "ResembleEnhance", "None"]
MIN_SILENCE_SEC = 0.8  # @param {type:"slider", min:0.2, max:2.0, step:0.05}
PADDING_SEC = 0.25  # @param {type:"slider", min:0.05, max:0.5, step:0.01}
CROSSFADE_MS = 30  # @param {type:"slider", min:10, max:100, step:5}
NORMALIZE_AUDIO = True  # @param {type:"boolean"}

# Export & Acceleration Flags
EXPORT_TIMELINE = True  # @param {type:"boolean"}
USE_GPU = True  # @param {type:"boolean"}
OVERWRITE = False  # @param {type:"boolean"}

# Assemble pipeline configuration dictionary
pipeline_config = {
    "denoise_engine": DENOISE_ENGINE,
    "min_silence_sec": float(MIN_SILENCE_SEC),
    "padding_sec": float(PADDING_SEC),
    "crossfade_ms": int(CROSSFADE_MS),
    "normalize_audio": bool(NORMALIZE_AUDIO),
    "export_timeline": bool(EXPORT_TIMELINE),
    "use_gpu": bool(USE_GPU),
    "overwrite": bool(OVERWRITE),
}

print("=" * 65)
print("ACTIVE PIPELINE CONFIGURATION:")
print(f"  * Input Directory       : {INPUT_DIR}")
print(f"  * Output Directory      : {OUTPUT_DIR}")
print(f"  * AI Denoise Engine     : {DENOISE_ENGINE}")
print(f"  * Min Silence to Cut    : {MIN_SILENCE_SEC:.2f}s (pauses < this are kept natural)")
print(f"  * Head/Tail Padding     : {PADDING_SEC * 1000:.0f}ms (breath and consonant buffer)")
print(f"  * Audio Crossfade       : {CROSSFADE_MS}ms (anti-pop cosine blend)")
print(f"  * Loudness Normalization: {NORMALIZE_AUDIO} (-14.0 LUFS EBU R128)")
print(f"  * Export NLE XML & EDL  : {EXPORT_TIMELINE} (FCP7 XML and CMX 3600 EDL)")
print(f"  * Hardware Encoding     : {USE_GPU} (NVIDIA NVENC / CPU fallback)")
print(f"  * Overwrite Outputs     : {OVERWRITE}")
print("=" * 65)


In [ ]:
# @title Step 4: Run Batch Processing Queue
import os
import sys
import time
import json
from interview_processor import process_batch

# Validate directory presence
active_input = INPUT_DIR
if not os.path.isdir(active_input):
    if os.path.isdir("./inputs"):
        active_input = "./inputs"
        print(f"[INFO] Colab path not found, using local folder: {active_input}")
    else:
        os.makedirs(active_input, exist_ok=True)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Commencing batch processing queue...")
print(f"   Input Path : {os.path.abspath(active_input)}")
print(f"   Output Path: {os.path.abspath(OUTPUT_DIR)}\n")

batch_start_time = time.time()
batch_results = process_batch(active_input, OUTPUT_DIR, pipeline_config)
total_elapsed_time = time.time() - batch_start_time

# Calculate summary metrics
successful_cuts = [r for r in batch_results if r.get("status") == "success"]
skipped_files = [r for r in batch_results if r.get("status") == "skipped"]
failed_files = [r for r in batch_results if r.get("status") == "failed"]

total_orig_duration = sum(r.get("stats", {}).get("original_duration", 0.0) for r in successful_cuts)
total_kept_duration = sum(r.get("stats", {}).get("kept_duration", 0.0) for r in successful_cuts)
total_silence_cut = sum(r.get("stats", {}).get("silence_removed", 0.0) for r in successful_cuts)
overall_silence_pct = (total_silence_cut / total_orig_duration * 100.0) if total_orig_duration > 0 else 0.0
speed_factor = (total_orig_duration / total_elapsed_time) if total_elapsed_time > 0 else 0.0

print("\n" + "=" * 75)
print("BATCH EXECUTION SUMMARY REPORT")
print("=" * 75)
print(f"* Total Videos Queued : {len(batch_results)}")
print(f"* Successfully Cut    : {len(successful_cuts)}")
print(f"* Skipped (Unchanged) : {len(skipped_files)}")
print(f"* Failed / Errors     : {len(failed_files)}")
print("-" * 75)
print(f"* Original Duration   : {total_orig_duration:.2f}s ({total_orig_duration / 60:.2f} minutes)")
print(f"* Clean Cut Duration  : {total_kept_duration:.2f}s ({total_kept_duration / 60:.2f} minutes)")
print(f"* Duration Saved      : {total_silence_cut:.2f}s ({total_silence_cut / 60:.2f} minutes)")
print(f"* Silence Cut Ratio   : {overall_silence_pct:.1f}%")
print(f"* Total Execution Time: {total_elapsed_time:.2f}s")
print(f"* Batch Speed Factor  : {speed_factor:.2f}x Realtime speed")
print("=" * 75)

if successful_cuts:
    print("\nPer-File Breakdown:")
    for item in successful_cuts:
        base = os.path.basename(item.get("file", ""))
        s = item.get("stats", {})
        t = item.get("processing_time_sec", 0.0)
        print(f"  * {base:<28} | {s.get('original_duration', 0):.1f}s -> {s.get('kept_duration', 0):.1f}s (-{s.get('silence_percentage', 0):.1f}%) | {t:.1f}s processing")


In [ ]:
# @title Step 5: Preview Results and Download Zip Archive
import os
import glob
import shutil
from IPython.display import display, Video

# 1. Preview first rendered clean-cut video
output_videos = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*_clean_cut.mp4")))
if output_videos:
    preview_target = output_videos[0]
    print(f"Interactive Player Preview: {os.path.basename(preview_target)}")
    try:
        display(Video(preview_target, embed=True, width=640, height=360))
    except Exception as err:
        print(f"Player embed info: {err}")
else:
    print("[INFO] No rendered video files found in output directory.")

# 2. Package outputs into a single .zip archive
archive_basename = "Interview_AI_Studio_Outputs"
archive_dest = os.path.join("/content", archive_basename) if os.path.exists("/content") else os.path.join(OUTPUT_DIR, "..", archive_basename)
zip_file_path = shutil.make_archive(archive_dest, "zip", OUTPUT_DIR)
archive_size_mb = os.path.getsize(zip_file_path) / (1024 * 1024)
print(f"\nResults archive bundled successfully: {zip_file_path} ({archive_size_mb:.2f} MB)")

# 3. Trigger one-click download in Google Colab
try:
    from google.colab import files
    print("Initiating browser download of ZIP archive...")
    files.download(zip_file_path)
except (ImportError, ModuleNotFoundError):
    print(f"[INFO] Local filesystem: Zip archive ready at {os.path.abspath(zip_file_path)}")
